# 
 Prétraitement des Données (Data Preprocessing Pipeline)

## Contexte et Objectifs
Le module de prétraitement a pour objectif de préparer et fiabiliser les données brutes issues de l'EDA pour alimenter les algorithmes d'apprentissage automatique sans fuite d'information (*data leakage*).

### Étapes du Pipeline :
1. **Chargement des données brutes**.
2. **Correction des incohérences logiques** (`illogic_value`) : Remplacement par NaN des valeurs impossibles (réseaux sociaux > temps d'écran).
3. **Détection des outliers** (`detecter_outlier`) : Identification par l'écart interquartile (IQR).
4. **Découpage des données** (`split_data`) : Split stratifié 80/20 et retrait des colonnes non informatives (`Student_ID`, `Age`).
5. **Construction du ColumnTransformer scikit-learn** (`preprocessing`) : Imputation, standardisation et encodages (One-Hot et Ordinal).
6. **Pipeline complet** (`preprocessign_data`).


## 1. Chargement des Données et Environnement
Importation des dépendances et chargement du jeu de données brut.


In [ ]:

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Importer les fonctions
from src.preprocessing import (
  
    illogic_value,
    detecter_outlier,
    split_data,
     preprocessing,
    preprocessign_data,
       
        
       
   
)

data_path = PROJECT_ROOT / "data" / "raw" / "student_mental_health.csv"
df=pd.read_csv(data_path)  
# Charger les données

## 2. Traitement des Incohérences Métier (`illogic_value`)
Remplacement par `np.nan` des valeurs impossibles où `Social_Media_Hours > Screen_Time_Hours`.
Ces valeurs manquantes générées seront ensuite proprement imputées lors de l'étape de transformation.


In [2]:
df = illogic_value(df)
print(f"Valeurs manquantes dans Social_Media_Hours après correction : {df['Social_Media_Hours'].isnull().sum()}")
df.head()


Valeurs manquantes dans Social_Media_Hours après correction : 95


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


## 3. Détection des Valeurs Aberrantes (`detecter_outlier`)
Application de la règle de Tukey (IQR) sur l'ensemble des variables numériques pour inspecter les valeurs extrêmes sans les supprimer de manière arbitraire.


In [3]:

numeric_cols = ["Sleep_Hours", "Study_Hours_Per_Week", "GPA", "Year_of_Study","Social_Media_Hours","Screen_Time_Hours","Financial_Stress","Family_Support","Attendance_Rate"]

for c in numeric_cols:

    outliers = detecter_outlier(df, c)

    print(f"\n========== {c} ==========")

    if outliers:
        for result in outliers:
            print("Colonne :", result["col"])
            print("Borne inférieure :", round(result["lower"], 2))
            print("Borne supérieure :", round(result["upper"], 2))
            print("Nombre d'outliers :", result["len outlier"])
            print("Valeurs détectées :")
            print(result["value_outlier"])
    else:
        print("Aucun outlier détecté")
    


========== Sleep_Hours ==========
Colonne : Sleep_Hours
Borne inférieure : 3.6
Borne supérieure : 10.0
Nombre d'outliers : 2
Valeurs détectées :
749     3.6
1982    3.3
Name: Sleep_Hours, dtype: float64

========== Study_Hours_Per_Week ==========
Colonne : Study_Hours_Per_Week
Borne inférieure : 4.0
Borne supérieure : 36.0
Nombre d'outliers : 7
Valeurs détectées :
340      3.9
453      3.2
519      3.6
716     38.3
866      2.4
937     37.5
1042     2.7
Name: Study_Hours_Per_Week, dtype: float64

========== GPA ==========
Colonne : GPA
Borne inférieure : 1.61
Borne supérieure : 3.5
Nombre d'outliers : 9
Valeurs détectées :
358     3.60
448     1.50
534     3.66
881     3.53
937     3.64
1299    3.57
1320    3.51
1541    1.60
1923    3.51
Name: GPA, dtype: float64

========== Year_of_Study ==========
Aucun outlier détecté

========== Social_Media_Hours ==========
Colonne : Social_Media_Hours
Borne inférieure : 2.95
Borne supérieure : 8.95
Nombre d'outliers : 19
Valeurs détectées :
22  

## 4. Découpage Entraînement / Test (`split_data`)
- Séparation des features $X$ et de la cible $y$ : La colonne d'identifiant `Student_ID` ainsi que `Age` sont écartées, et la cible `Mental_Health_Risk` est encodée numériquement (`Low=0`, `Medium=1`, `High=2`).
- Découpage en 80% Train et 20% Test avec stratification pour préserver la proportion exacte des classes.


In [4]:
X_train, X_test, y_train, y_test = split_data(df)
print(f"Dimensions X_train : {X_train.shape}")
print(f"Dimensions X_test  : {X_test.shape}")
print(f"Dimensions y_train : {y_train.shape}")
print(f"Dimensions y_test  : {y_test.shape}")


Dimensions X_train : (1600, 14)
Dimensions X_test  : (400, 14)
Dimensions y_train : (1600,)
Dimensions y_test  : (400,)


## 5. Pipeline de Prétraitement Global (`preprocessign_data`)
Exécution de la chaîne complète de préparation :
- `pipeline_num` : Imputation par médiane + StandardScaler.
- `pipeline_onehot` & `nominal_pipeline` : Imputation par mode + OneHotEncoder.
- `pipeline_ordinal` : Imputation par mode + OrdinalEncoder avec respect de l'ordre hiérarchique.
Retourne les jeux de données d'entraînement, de test, et l'objet `ColumnTransformer`.


In [5]:

X_train, X_test, y_train, y_test, preprocessor = preprocessign_data(df)

print(f"Dimensions X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"Dimensions y_train : {y_train.shape} | y_test : {y_test.shape}")

X_train_processed = preprocessor.fit_transform(X_train)
print(f" Dimensions après preprocessing : {X_train_processed.shape}")
print(f"Valeurs NaN restantes après imputation : {np.isnan(X_train_processed).sum()}")

# Conversion en DataFrame pour afficher le résultat nettoyé et sans NaN
feature_names = preprocessor.get_feature_names_out()
df_processed = pd.DataFrame(X_train_processed, columns=feature_names)

# Aperçu propre des 5 premières lignes transformées (SANS AUCUN NaN)
df_processed.head()


Dimensions X_train : (1600, 14) | X_test : (400, 14)
Dimensions y_train : (1600,) | y_test : (400,)
✅ Dimensions après preprocessing : (1600, 18)
✅ Valeurs NaN restantes après imputation : 0


,num__Year_of_Study,num__Study_Hours_Per_Week,num__Sleep_Hours,num__Screen_Time_Hours,num__Family_Support,num__Financial_Stress,num__Attendance_Rate,num__GPA,num__Social_Media_Hours,binary__Part_Time_Job_No,binary__Part_Time_Job_Yes,binary__Previous_Mental_Health_History_No,binary__Previous_Mental_Health_History_Yes,nominale__Gender_Female,nominale__Gender_Male,nominale__Gender_Non-binary,ordinal__Physical_Activity,ordinal__Diet_Quality
0,-1.224199,-1.462077,-0.042057,-0.089959,-0.021546,-0.259617,-0.520291,-0.852807,-0.096295,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0
1,-1.224199,-1.375823,1.227579,-0.391708,-1.233064,0.657234,-1.164891,-1.696649,-0.722972,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
2,1.058690,0.728769,-0.495498,0.634238,0.694351,0.225775,1.556754,-0.340475,0.082755,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0
3,-0.463236,-0.116518,0.411385,-0.210658,-0.627305,-0.637144,-0.907051,-0.250063,0.351330,0.0,1.0,1.0,0.0,0.0,1.0,0.0,2.0,1.0
4,1.058690,0.142243,-0.314122,-0.029609,1.189972,1.304423,-3.256259,-1.214454,-0.364871,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,2.0


## 6. Conclusion du Prétraitement

### Bilan technique :
- **Intégrité garantie** : Pas de fuite de données (*data leakage*), le `ColumnTransformer` est conçu pour être ajusté sur le train set et appliqué sans biais au test set.
- **Robustesse** : Gestion native des valeurs manquantes et des valeurs inconnues lors de l'inférence.
- **Modularité** : Le pipeline est prêt à être inséré dans une chaîne scikit-learn (`Pipeline`) combinant transformation et classification.
